In [ ]:
#video

In [ ]:
from ultralytics import YOLO
import cv2

# Load the custom YOLOv8 model (update path if needed)
model_path = 'best_2025.pt'  # Your model file
model = YOLO(model_path)

# Open webcam (source=0 for default camera; change to video file path if needed)
cap = cv2.VideoCapture('safety.mp4')

if not cap.isOpened():
    print("Error: Could not open video source.")
    exit()

print("Starting real-time detection. Press 'q' to quit.")

while True:
    # Read frame from camera
    ret, frame = cap.read()
    if not ret:
        print("Error: Failed to capture frame.")
        break
    
    # Run YOLO inference on the frame
    results = model(frame, verbose=False)  # verbose=False for cleaner output
    
    # Draw results on the frame
    annotated_frame = results[0].plot()  # Draws boxes, labels, and confidences
    
    # Display the frame
    cv2.imshow('Real-Time Object Detection', annotated_frame)
    
    # Break loop on 'q' key press
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Cleanup
cap.release()
cv2.destroyAllWindows()
print("Detection stopped.")

In [ ]:
#video file

In [ ]:
import cv2
from ultralytics import YOLO
import sys

# 1. Load Model
# Ensure '2026.pt' is in the same folder, or use 'yolov8n.pt' for testing
try:
    model = YOLO('321.pt') 
except Exception as e:
    print(f"Error loading model: {e}")
    print("Make sure '2026.pt' is in the correct path.")
    sys.exit(1)

# 2. Open Webcam with Windows DirectShow (Fixes 'Could not open video source')
# We explicitly ask Windows to use the DirectShow backend
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)

# If index 0 fails, try index 1 automatically
if not cap.isOpened():
    print("Camera 0 failed. Trying Camera 1...")
    cap = cv2.VideoCapture(1, cv2.CAP_DSHOW)

if not cap.isOpened():
    print("CRITICAL ERROR: Could not open any video source.")
    print("1. Check if Zoom/Teams is holding the camera.")
    print("2. Check Windows Privacy settings.")
    sys.exit(1)

print("Starting real-time detection. Press 'q' to quit.")

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            print("Error: Failed to capture frame.")
            break
        
        # 3. Inference
        results = model(frame, verbose=False)
        
        # 4. Visualization
        annotated_frame = results[0].plot()
        
        # 5. Display (Wrapped in try-block in case of GUI errors)
        try:
            cv2.imshow('Real-Time Object Detection', annotated_frame)
        except cv2.error as e:
            print("GUI Error: Your OpenCV installation doesn't support windows.")
            print("Solution: pip uninstall opencv-python-headless -> pip install opencv-python")
            break
        
        # Break loop on 'q' key press
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

except KeyboardInterrupt:
    print("Interrupted by user.")

finally:
    # 6. Safe Cleanup
    cap.release()
    try:
        cv2.destroyAllWindows()
    except:
        pass
    print("Detection stopped.")

In [ ]:
import cv2
from ultralytics import YOLO
import sys
import time

# --- CONFIGURATION ---
MODEL_FILE = '321.pt'  # Assurez-vous que c'est le bon nom (ex: 'best.pt' ou 'yolov8n.pt')
# ---------------------

print("🔍 RECHERCHE AUTOMATIQUE DE LA CAMÉRA DROIDCAM...")

working_cap = None
found_index = -1

# On teste les index de 0 à 3 pour trouver DroidCam
for i in range(4):
    print(f"👉 Test de la caméra {i}...")
    # On utilise CAP_DSHOW pour Windows
    temp_cap = cv2.VideoCapture(i, cv2.CAP_DSHOW)
    
    if temp_cap.isOpened():
        # On laisse un peu de temps à la caméra pour s'initialiser
        time.sleep(1)
        ret, frame = temp_cap.read()
        
        # Si on arrive à lire une image et qu'elle n'est pas vide
        if ret and frame is not None and frame.size > 0:
            print(f"✅ SUCCÈS ! Caméra active trouvée à l'index {i}")
            working_cap = temp_cap
            found_index = i
            break
        else:
            print(f"   ❌ Caméra {i} ouverte mais écran noir (Probablement la caméra MSI cassée).")
            temp_cap.release()
    else:
        print(f"   ❌ Aucune caméra à l'index {i}.")

if working_cap is None:
    print("\n🛑 ERREUR CRITIQUE : Aucune caméra fonctionnelle trouvée.")
    print("Vérifiez :")
    print("1. L'application DroidCam sur le PC est bien sur 'START' ?")
    print("2. L'application DroidCam sur le téléphone est allumée ?")
    sys.exit(1)

# --- DÉMARRAGE DE YOLO ---
print(f"\n🚀 Démarrage de la détection sur la caméra {found_index}...")

try:
    # Chargement du modèle
    try:
        model = YOLO(MODEL_FILE)
    except:
        print(f"Modèle '{MODEL_FILE}' introuvable, utilisation de 'yolov8n.pt'...")
        model = YOLO('yolov8n.pt')

    while True:
        ret, frame = working_cap.read()
        
        if not ret:
            print("Perte de connexion caméra.")
            break
            
        # Détection
        results = model(frame, verbose=False, stream=True)
        
        # Affichage
        for result in results:
            annotated_frame = result.plot()
            cv2.imshow(f'Detection YOLO (Camera {found_index})', annotated_frame)
        
        # Quitter avec 'q'
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

except Exception as e:
    print(f"Erreur pendant l'exécution : {e}")

finally:
    working_cap.release()
    cv2.destroyAllWindows()
    print("Programme terminé.")

In [ ]:
from ultralytics import YOLO
model = YOLO('321.pt') # Use your model file name
print(model.names)

In [ ]:
import cv2
from ultralytics import YOLO
import sys

# 1. Load Model
try:
    # Use 'best_2025.pt' (or '321.pt' if you prefer)
    model = YOLO('best_2025.pt') 
    
    # --- AUTOMATIC LABEL FIX ---
    # This checks how many classes the model has and ensures they all have names.
    # This prevents the 'KeyError' you just saw.
    existing_names = model.model.names
    custom_names = {0: 'Fire', 1: 'Smoke', 2: 'Fire', 3: 'Fire'} # Map 2 and 3 to 'Fire' just in case
    
    # Fill in any missing indexes so the code doesn't crash
    for i in range(20): # Covers up to 20 possible detection classes
        if i not in custom_names:
            custom_names[i] = f"Object_{i}"
            
    model.model.names = custom_names
    print("Model loaded and labels synchronized.")
except Exception as e:
    print(f"Error loading model: {e}")
    sys.exit(1)

# 2. Open Camera (Phone App)
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
if not cap.isOpened():
    cap = cv2.VideoCapture(1, cv2.CAP_DSHOW)

if not cap.isOpened():
    print("CRITICAL ERROR: Could not open camera. Check DroidCam/iVCam connection.")
    sys.exit(1)

print("Starting detection. Press 'q' on the VIDEO WINDOW to quit.")

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        # 3. Inference
        # Increased confidence to 0.5 to reduce "fake" detections
        results = model(frame, conf=0.5, verbose=False)
        
        # 4. Visualization & Display
        # The loop ensures we handle the results correctly
        for r in results:
            annotated_frame = r.plot()
            cv2.imshow('Fire Detection - Phone Camera', annotated_frame)
        
        # Press 'q' to quit
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

except Exception as e:
    print(f"An error occurred: {e}")

finally:
    cap.release()
    cv2.destroyAllWindows()
    print("Program closed safely.")

Model loaded and labels synchronized.
Starting detection. Press 'q' on the VIDEO WINDOW to quit.
